In [1]:
import os
import pandas as pd
from maomao.parsing.parsing_utils import *
from maomao.utils.constants import *

#### Processing and standardizing peptide datasets (tAMPer)

This notebook processes and standardizes the **tAMPer** dataset by integrating peptide sequences derived from both FASTA files and experimentally resolved protein structures. Toxicity labels are inferred from sequence identifiers, followed by duplicate resolution, metadata generation, and dataset export for downstream machine learning analyses.

- **Toxic effect / endpoint:** toxic
- **Source:** tAMPer
- **Sequence scope:** only non-modified peptide sequences are retained for the final dataset.

The pipeline performs the following steps:

- **Parses peptide sequences from FASTA files**:
  - including positive, and negative sequence collections.
- **Extracts peptide sequences from structural PDB files**:
  - sequences are recovered at the chain level from non-standard PDB formats.
- **Integrates sequence- and structure-derived peptides** into a unified dataset.
- **Infers toxicity labels from sequence identifiers**:
  - sequences starting with `toxic` are labeled as toxic (1),
  - sequences starting with `non_toxic` are labeled as non-toxic (0).
- **Performs duplicate sequence checks**:
  - merges identical sequences with consistent labels,
  - flags conflicting annotations as erroneous.
- **Generates dataset-level metadata** using a centralized raw data description file.
- **Exports curated outputs**:
  - `processed_toxic_dataset.csv`
  - `detected_error_sequences.csv`
  - `metadata.json`

In [2]:
name_source = "tAMPer"
name_task = "toxic_effect_classification"

# PATH_INPUT and PATH_EXPORT are imported from maomao.utils.constants
# Update them in constants.py according to the required input and export paths.

- Reading raw data

In [3]:
folders = [
    f"{PATH_INPUT}/{name_source}/sequences/",
    f"{PATH_INPUT}/{name_source}/sequences/negative", 
    f"{PATH_INPUT}/{name_source}/sequences/positive"
]

df_sequences = pd.concat([
    read_fasta_doc(os.path.join(folder, file))
    for folder in folders
    for file in os.listdir(folder)
    if os.path.isfile(os.path.join(folder, file))
])

In [4]:
pdb_dir = f"{PATH_INPUT}/{name_source}/structures"
records = []

for pdb_file in Path(pdb_dir).glob("*.pdb"):
    sequences = extract_sequence_from_pdb_notstandar(str(pdb_file))

    for chain_id, seq in sequences.items():
        records.append({
            "id": pdb_file.stem,
            "sequence": seq
        })

df_structures = pd.DataFrame(records)

- Concatenate dataset

In [5]:
df_tamper = pd.concat([
    df_sequences,
    df_structures],
    ignore_index=True
)

In [6]:
df_tamper["label"] = pd.NA
df_tamper.loc[df_tamper["id"].str.lower().str.startswith("non_toxic"), "label"] = 0
df_tamper.loc[df_tamper["id"].str.lower().str.startswith("toxic"), "label"] = 1
df_tamper = df_tamper[["sequence", "label"]]
df_tamper.shape

(5978, 2)

- Checking duplicates

In [7]:
df_remove_duplicated, df_errors, df_unique = processing_duplicated(df_tamper, group_seq="sequence", sort_key="label")
df_full = pd.concat([df_unique, df_remove_duplicated], axis=0)

In [8]:
df_full.shape

(5708, 2)

In [9]:
df_errors.shape

(55, 1)

- Working with metada

In [10]:
df_metada = read_metadata("../../raw_data/raw_data_description.xlsx", name_source)
dict_metadata = create_metada_with_multiple_values(df_metada)

In [11]:
dict_metadata.update({
    "number_of_raw_sequences": int(len(df_tamper)),
    "number_of_sequences_retained": len(df_full),
    "number_of_positive_sequences": int((df_full["label"] == 1).sum()),
    "number_of_negative_sequences": int((df_full["label"] == 0).sum()),
    "number_of_erroneous_sequences": int(len(df_errors)),
    "modified_sequences_included": False,
})

dict_metadata

{'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'GNU general public license',
 'year of publication': 2024,
 'last update date': datetime.datetime(2023, 11, 19, 0, 0),
 'download date': Timestamp('2024-08-01 00:00:00'),
 'file format': 'fasta;PDB',
 'peptide property': 'toxic',
 'dataset information': 'Negative;Positive',
 'unit of measurement': 'No information',
 'obtaining negative dataset': '"Characteristic threshold (IC50, MIC, etc.)";No information',
 'repository or server': 'https://github.com/bcgsc/tAMPer',
 'publication': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC11193153/#pro5076-sec-0028',
 'number_of_raw_sequences': 5978,
 'number_of_sequences_retained': 5708,
 'number_of_positive_sequences': 1790,
 'number_of_negative_sequences': 3918,
 'number_of_erroneous_sequences': 55,
 'modified_sequences_included': False}

- Exporting data

In [12]:
os.makedirs(f"{PATH_EXPORT}/{name_task}/{name_source}/", exist_ok=True)
export_json(f"{PATH_EXPORT}/{name_task}/{name_source}/metadata.json", dict_metadata)

In [13]:
df_full.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_toxic_dataset.csv", index=False)
df_errors.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/detected_error_sequences.csv", index=False)